In [ ]:
import os
from dotenv import load_dotenv

from langsmith import Client
from langsmith.evaluation import evaluate

# Make sure src/ is on the path
import sys
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

#Add the 'src' folder to Python's path
#This tells Python: "Look inside Project_Root/src for modules"
src_path = os.path.join(project_root, 'src')

if src_path not in sys.path:
    sys.path.append(src_path)

from backend.src.graph import run_email_agent

load_dotenv()

client = Client()
print("✅ LangSmith client ready")


✅ LangSmith client ready


In [3]:
# List datasets to confirm your Golden_DataSet exists
datasets = list(client.list_datasets())
[(d.name, d.id) for d in datasets]


[('Golden_DataSet-2', UUID('038c851c-ce35-4147-8054-efdc91a21629')),
 ('Golden_DataSet', UUID('e050aea3-76cb-4a8d-a03b-fe76b9e16bae')),
 ('Agent_Math_Dataset', UUID('67f6f390-8052-48e5-85ef-b44b37658050')),
 ('Evaluator: Agent:Agent-Success-Judge (5814d026) – correctness',
  UUID('ad92551c-d287-4c8f-8b35-c5ce64530ecc')),
 ('Evaluator: Agent-Grader (28e171bf) – correctness',
  UUID('ffa3c9c7-4a10-4327-a333-1e2768959c55')),
 ('Evaluator: langsmith-demo:Correctness-Judge (c5f5085e) – correctness',
  UUID('ee4b293c-e0b8-424b-8452-258f88c9e508'))]

In [4]:
def eval_wrapper(inputs):
    """
    example.inputs must contain 'subject' and 'body'
    as configured in the LangSmith dataset.
    """
    subject = inputs["subject"]
    body = inputs["body"]

    result = run_email_agent(subject=subject, body=body)

    return {
        # This is what the judge sees as {{model_output}}
        "model_output": result["reply"],
        # Optional extra field to compare with triage_label
        "triage_prediction": result["triage"],
    }


In [ ]:
from itertools import islice

dataset_name = "Golden_DataSet-2"  


dataset = client.read_dataset(dataset_name="Golden_DataSet-2")
sample_examples = list(islice(client.list_examples(dataset_id=dataset.id), 5))

for ex in sample_examples:
    print("=== EMAIL ===")
    print("Subject:", ex.inputs["subject"])
    print("Body:", ex.inputs["body"])
    out = eval_wrapper(ex.inputs)
    print("Model output (first 200 chars):", (out["model_output"] or "")[:200])
    print("Pred triage:", out.get("triage_prediction"))
    print()


=== EMAIL ===
Subject: Vendor asking for meeting to demo tool
Body: We’d love to schedule a short demo of our monitoring platform. Are you available for a 30-minute call next week?
✅ Triage: notify-human
Notifying human: important email.
Model output (first 200 chars): 
Pred triage: notify-human

=== EMAIL ===
Subject: Travel change request
Body: Can you move my return flight from March 18 to March 20 instead? Let me know if there are any fees.
✅ Triage: notify-human
Notifying human: important email.
Model output (first 200 chars): 
Pred triage: notify-human

=== EMAIL ===
Subject: GitHub notifications
Body: You have 15 new notifications on GitHub including issue comments and PR reviews. Visit GitHub to see the details.
✅ Triage: notify-human
Notifying human: important email.
Model output (first 200 chars): 
Pred triage: notify-human

=== EMAIL ===
Subject: Manager: Requesting weekly summary
Body: Hi, starting next week, can you send me a short summary every Friday of what the team acc

In [6]:
results = evaluate(
    eval_wrapper,
    data=dataset_name,
    evaluators=[],   # exact evaluator name from LangSmith UI
    experiment_prefix="email-bot-v2",
)

results


View the evaluation results for experiment: 'email-bot-v2-42f423ff' at:
https://smith.langchain.com/o/053cdddb-ecb9-47b0-84bc-3f3b9370e929/datasets/038c851c-ce35-4147-8054-efdc91a21629/compare?selectedSessions=7e35b401-b46b-48c6-a9c0-d5a439a50857




d:\Aayush\College\Interships\Infosys-Building an Ambient Agent\langgraph-email-assistant\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
1it [00:02,  2.44s/it]

✅ Triage: notify-human
Notifying human: important email.


Error running target function: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 27.434413577s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProject

✅ Triage: notify-human
Notifying human: important email.


4it [00:51, 11.18s/it]

✅ Triage: ignore
Ignored email (newsletter/promo).


4it [00:52, 13.02s/it]


,inputs.body,inputs.subject,outputs.model_output,outputs.triage_prediction,error,reference.triage_label,reference.ideal_response,execution_time,example_id,id,outputs.output
0,We detected a login to your account from a new...,Suspicious login detected on your account,NaN,notify-human,None,notify-human,Flag this as a security-related email that mus...,2.435384,60b6760f-af29-484b-99c1-9b895cc0f02f,019b3c94-f331-7e82-9a7b-c93266c216e3,NaN
1,We are planning a casual team lunch next Wedne...,Team lunch next week,NaN,NaN,"ChatGoogleGenerativeAIError(""Error calling mod...",respond-act,"Reply in a friendly tone, confirm attendance i...",36.366708,86043e47-0b28-4233-89bf-a2e2bc0c6444,019b3c94-fcb7-78e3-97cb-6c7060c3d410,NaN
2,"Hi, could we schedule a 60-minute kickoff meet...",Meeting request for project kickoff,NaN,notify-human,None,respond-act,Politely propose a specific time slot next wee...,9.948339,ca1c9663-2dd8-4012-8144-ba5ee02329a2,019b3c95-8acc-7f80-afbb-248596b3841c,NaN
3,Welcome to this week in tech! We cover the lat...,Newsletter: This week in tech,NaN,ignore,None,ignore,Mark as non-actionable informational content a...,2.696098,d398a203-5e24-485a-9aa0-cf9ac583f363,019b3c95-b1ae-7863-8e4d-e90e0b74e750,NaN
